# MyAI K1 — Data: 10M FineWeb + 32k BPE + pack-512 (CPU-only kernel)
No GPU needed: streaming + BPE + packing are CPU work. Outputs (tokenizer-32k, packed shards, manifests) persist as kernel output for K2/K3.

In [ ]:
!git clone https://github.com/aloksingh9667/llm.git myai 2>&1 | tail -1
%cd /kaggle/working/myai
!pip install -q -r requirements/train.txt 2>&1 | tail -1
!python scripts/check_env.py

In [ ]:
# ~10M-token FineWeb sample (ODC-By-1.0). Exact token count measured post-hoc with the 32k tokenizer below.
!python scripts/stream_fineweb.py --max-tokens 10000000 --out data/raw/fineweb-10M.txt
!python scripts/prepare_tokenizer_corpus.py

In [ ]:
# 32k BPE via indexed trainer (identical merges, much faster). This is the long cell.
!python scripts/train_tokenizer.py --corpus data/tokenizer_corpus/train.txt --vocab-size 32000 --out data/tokenizer_corpus/tokenizer-32k.json --fast
!python scripts/eval_tokenizer.py --tokenizer data/tokenizer_corpus/tokenizer-32k.json --corpus data/tokenizer_corpus/train.txt --report /kaggle/working/tokenizer-32k-eval.json

In [ ]:
%%writefile configs/data-pack-fw10m.yaml
version: "1.0.0"
corpus_path: data/tokenizer_corpus/train.txt
tokenizer_path: data/tokenizer_corpus/tokenizer-32k.json
output_dir: data/processed/fw10m-32k
seq_len: 512
val_ratio: 0.01

In [ ]:
!python scripts/prepare_sequences.py --config configs/data-pack-fw10m.yaml --rows-per-shard 2048

In [ ]:
!cp data/tokenizer_corpus/tokenizer-32k.json /kaggle/working/
!cp data/processed/fw10m-32k/manifest.json /kaggle/working/pack-manifest.json
!cp data/processed/fw10m-32k/index.json /kaggle/working/pack-index.json
!cp data/tokenizer_corpus/manifest.json /kaggle/working/corpus-manifest.json
!ls -la /kaggle/working/ && python -c "import json; m=json.load(open('/kaggle/working/pack-manifest.json')); print('tokens:', m['total_tokens'], 'train:', m['train_sequences'], 'val:', m['val_sequences'])"